In [63]:
import sys
print(sys.executable)
print(sys.version)  

/Users/yadirarojas/Documents/GitHub/Hospital-Readmission-Penalties/.venv/bin/python
3.12.7 (v3.12.7:0b05ead877f, Sep 30 2024, 23:18:00) [Clang 13.0.0 (clang-1300.0.29.30)]


In [64]:
from pathlib import Path 
print(Path.cwd())

/Users/yadirarojas/Documents/GitHub/Hospital-Readmission-Penalties/notebooks


In [65]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
print(list(RAW.iterdir()))
readm = pd.read_csv(RAW/ "FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv", dtype=str)

[PosixPath('../data/raw/FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv'), PosixPath('../data/raw/README.md'), PosixPath('../data/raw/Hospital_General_Information.csv')]


## 1. Hospital Readmission Reduction Program Dataset

In [66]:
import pandas as pd

RAW = Path("../data/raw")
readm = pd.read_csv(RAW/ "FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv", dtype=str)

print(f"Dataset shape: {readm.shape[0]} row, {readm.shape[1]} columns")
print(readm["Facility ID"].nunique())
print("Column Names: ", readm.columns.tolist()) 
readm.head(6)

Dataset shape: 18330 row, 12 columns
3055
Column Names:  ['Facility Name', 'Facility ID', 'State', 'Measure Name', 'Number of Discharges', 'Footnote', 'Excess Readmission Ratio', 'Predicted Readmission Rate', 'Expected Readmission Rate', 'Number of Readmissions', 'Start Date', 'End Date']


,Facility Name,Facility ID,State,Measure Name,Number of Discharges,Footnote,Excess Readmission Ratio,Predicted Readmission Rate,Expected Readmission Rate,Number of Readmissions,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,06/30/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-CABG-HRRP,137,NaN,0.9531,10.3960,10.9078,13,07/01/2021,06/30/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-AMI-HRRP,273,NaN,0.9370,13.2998,14.1948,33,07/01/2021,06/30/2024
3,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-COPD-HRRP,122,NaN,0.9823,16.6384,16.9389,19,07/01/2021,06/30/2024
4,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-PN-HRRP,507,NaN,0.9871,15.7529,15.9591,79,07/01/2021,06/30/2024
5,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-HF-HRRP,653,NaN,1.0233,20.5695,20.1010,136,07/01/2021,06/30/2024


In [67]:
#1. How many rows does each hospital have?
rows_per_hospital = readm.groupby("Facility ID").size() # will hold how many times a "facility id" occured reperesnting the rows
print(rows_per_hospital.value_counts()) # will count and display the frequency of the values when grouped allows us to see if there were hospitals who didnt have exact 6 rows as hypothesized

#2. How many distinct measures, and what are they?
print(readm["Measure Name"].nunique()) # Gets the count of how many different Measure names there is
print(readm["Measure Name"].unique()) # Prints each measure name once

#3. Is each (hospital, meausre) pair unique?
print(readm.duplicated(subset=["Facility ID", "Measure Name"]).sum())

#4. Total Hospitals
print(readm["Facility ID"].nunique(), "hospitals,", len(readm), "rows")

6    3055
Name: count, dtype: int64
6
<StringArray>
['READM-30-HIP-KNEE-HRRP',     'READM-30-CABG-HRRP',      'READM-30-AMI-HRRP',
     'READM-30-COPD-HRRP',       'READM-30-PN-HRRP',       'READM-30-HF-HRRP']
Length: 6, dtype: str
0
3055 hospitals, 18330 rows


In [68]:
ids = ["Facility Name", "Facility ID", "State", "Measure Name"]
for c in [c for c in readm.columns if c not in ids]:
    nonnum = readm[c][pd.to_numeric(readm[c], errors="coerce").isna()]
    print(c, "->", nonnum.value_counts(dropna=False).head(5).to_dict())

Number of Discharges -> {nan: 10088}
Footnote -> {nan: 11343}
Excess Readmission Ratio -> {nan: 6610}
Predicted Readmission Rate -> {nan: 6610}
Expected Readmission Rate -> {nan: 6610}
Number of Readmissions -> {nan: 6610, 'Too Few to Report': 3683}
Start Date -> {'07/01/2021': 18330}
End Date -> {'06/30/2024': 18330}


In [69]:
ratio = pd.to_numeric(readm["Excess Readmission Ratio"], errors="coerce")
print(ratio.groupby(readm["Measure Name"]).agg(total="size", usable="count"))

                        total  usable
Measure Name                         
READM-30-AMI-HRRP        3055    1736
READM-30-CABG-HRRP       3055     878
READM-30-COPD-HRRP       3055    2323
READM-30-HF-HRRP         3055    2621
READM-30-HIP-KNEE-HRRP   3055    1447
READM-30-PN-HRRP         3055    2715


## 5. Hospital General Information Dataset

In [70]:
gen = pd.read_csv(RAW / "Hospital_General_Information.csv", dtype = str)
print(f"Data shape: {gen.shape[0]} row, {gen.shape[1]} columns")
print(gen["Facility ID"].is_unique) # Checks all IDs are unique no duplicate == TRUE
print(gen["Hospital Ownership"].value_counts(dropna=False)) # 
print(readm["Facility ID"].str.len().value_counts(), gen["Facility ID"].str.len().value_counts())

Data shape: 5419 row, 38 columns
True
Hospital Ownership
Voluntary non-profit - Private                 2322
Proprietary                                    1063
Government - Hospital District or Authority     512
Government - Local                              393
Voluntary non-profit - Other                    353
Voluntary non-profit - Church                   264
Government - State                              209
Veterans Health Administration                  132
Physician                                        80
Government - Federal                             42
Department of Defense                            32
Tribal                                           17
Name: count, dtype: int64
Facility ID
6    18330
Name: count, dtype: int64 Facility ID
6    5419
Name: count, dtype: int64


In [71]:
r_ids, g_ids = set(readm["Facility ID"]), set(gen["Facility ID"])
print(len(r_ids - g_ids), "in readmissins but not in general info")
print(len(g_ids - r_ids), "in general info but not in readmissions")

20 in readmissins but not in general info
2384 in general info but not in readmissions


## 6.Value Audit
* **Understand the 2384 that are in Gen. Info. but not in readmissions**
* **Hosp Info accounts for all types of Medicare-Reg Hosp. while HRRP readm. coversonly a subset**

* **View the 20 that are in readmission but not in general***

In [72]:
gen["in_hrrp"] = gen["Facility ID"].isin(r_ids)
print(pd.crosstab(gen["Hospital Type"], gen["in_hrrp"]))
print()
print(pd.crosstab(gen["Hospital Ownership"], gen["in_hrrp"]))

in_hrrp                               False  True 
Hospital Type                                     
Acute Care - Department of Defense       32      0
Acute Care - Veterans Administration    132      0
Acute Care Hospitals                     72   3035
Childrens                                94      0
Critical Access Hospitals              1382      0
Long-term                                 5      0
Psychiatric                             624      0
Rural Emergency Hospital                 43      0

in_hrrp                                      False  True 
Hospital Ownership                                       
Department of Defense                           32      0
Government - Federal                            19     23
Government - Hospital District or Authority    292    220
Government - Local                             258    135
Government - State                             164     45
Physician                                       10     70
Proprietary              

In [73]:
# Breakdown of the 72 acute care hospitals not in HRRP

not_in_hrrp = gen[(gen["Hospital Type"] == "Acute Care Hospitals") & (gen["in_hrrp"] == False)]
state_bd = not_in_hrrp.groupby("State").size().sort_values(ascending=False)
print(state_bd)

State
PR    53
TX     5
AZ     2
GA     2
GU     2
VI     2
AS     1
CA     1
MP     1
UT     1
VA     1
WY     1
dtype: int64


In [74]:
unmatched = sorted(r_ids - g_ids)
readm[readm["Facility ID"].isin(unmatched)][["Facility ID", "Facility Name", "State"]].drop_duplicates()

,Facility ID,Facility Name,State
24,010008,CRENSHAW COMMUNITY HOSPITAL,AL
48,010018,CALLAHAN EYE HOSPITAL,AL
180,010059,LAWRENCE MEDICAL CENTER,AL
348,010120,MONROE COUNTY HOSPITAL,AL
1032,040067,MAGNOLIA REGIONAL MEDICAL HOSPITAL,AR
1524,050125,REGIONAL MEDICAL CENTER OF SAN JOSE,CA
2400,050589,UCI HEALTH - PLACENTIA LINDA,CA
3174,070012,ROCKVILLE GENERAL HOSPITAL,CT
3516,100047,SHOREPOINT HEALTH PUNTA GORDA,FL
3666,100092,ORLANDO HEALTH ROCKLEDGE HOSPITAL,FL


In [75]:
um= readm.loc[readm["Facility ID"].isin(unmatched), ["Facility ID", "Facility Name", "State"]].drop_duplicates()
for _, r in um.iterrows():
    key = max(r["Facility Name"].replace(",", "").split(), key=len)
    hits = gen[(gen["State"] == r["State"]) & gen["Facility Name"].str.contains(key, case=False, na=False, regex=False)]
    print(r["Facility ID"], r["Facility Name"], "->", hits[["Facility ID", "Facility Name"]].values.tolist()[:3])

010008 CRENSHAW COMMUNITY HOSPITAL -> [['010034', 'COMMUNITY HOSPITAL INC'], ['010052', 'LAKE MARTIN COMMUNITY HOSPITAL'], ['010097', 'ELMORE COMMUNITY HOSPITAL']]
010018 CALLAHAN EYE HOSPITAL -> []
010059 LAWRENCE MEDICAL CENTER -> []
010120 MONROE COUNTY HOSPITAL -> [['010007', 'MIZELL MEMORIAL HOSPITAL'], ['010019', 'HELEN KELLER HOSPITAL'], ['010024', 'JACKSON HOSPITAL & CLINIC INC']]
040067 MAGNOLIA REGIONAL MEDICAL HOSPITAL -> []
050125 REGIONAL MEDICAL CENTER OF SAN JOSE -> [['050045', 'EL CENTRO REGIONAL MEDICAL CENTER'], ['050060', 'COMMUNITY REGIONAL MEDICAL CENTER'], ['050082', 'ST JOHNS REGIONAL MEDICAL CENTER']]
050589 UCI HEALTH - PLACENTIA LINDA -> []
070012 ROCKVILLE GENERAL HOSPITAL -> []
100047 SHOREPOINT HEALTH PUNTA GORDA -> []
100092 ORLANDO HEALTH ROCKLEDGE HOSPITAL -> []
240052 LAKE REGION HEALTHCARE CORPORATION -> []
260176 ST LUKE'S DES PERES HOSPITAL -> [['260001', 'MERCY HOSPITAL JOPLIN'], ['260011', "SSM HEALTH ST. MARY'S HOSPITAL - JEFFERSON CITY"], ['26002

In [76]:
merged = readm.merge(
    gen[["Facility ID", "Hospital Ownership", "Hospital Type"]],
    on="Facility ID", how="left", validate="many_to_one",
)
print(len(merged))   # must equal 18,330 for all hospitals in readmission dataset

18330


In [77]:
readm["has_ratio"] = pd.to_numeric(readm["Excess Readmission Ratio"], errors="coerce").notna()
print(readm.groupby("Facility ID")["has_ratio"].sum().value_counts().sort_index())

has_ratio
0    222
1    202
2    239
3    484
4    599
5    662
6    647
Name: count, dtype: int64


In [78]:
print(readm[readm["Facility ID"].isin(unmatched)]["has_ratio"].sum(), "usable rows of 120")

39 usable rows of 120


In [79]:
raw = pd.read_csv(RAW / "FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv", dtype=str, keep_default_na=False)
for c in ["Excess Readmission Ratio", "Number of Discharges", "Number of Readmissions", "Footnote"]:
    bad = raw[c][pd.to_numeric(raw[c], errors="coerce").isna()]
    print(c, bad.value_counts().head(8).to_dict())

Excess Readmission Ratio {'N/A': 6610}
Number of Discharges {'N/A': 10088}
Number of Readmissions {'N/A': 6610, 'Too Few to Report': 3683}
Footnote {'': 11343}


In [80]:
raw["has_ratio"] = pd.to_numeric(raw["Excess Readmission Ratio"], errors="coerce").notna()
raw["has_disch"] = pd.to_numeric(raw["Number of Discharges"], errors="coerce").notna()
print(pd.crosstab(raw["has_ratio"], raw["has_disch"]))
print()
print(pd.crosstab(raw["Footnote"], raw["has_ratio"]))
print()
print(pd.crosstab(raw["Number of Readmissions"] == "Too Few to Report", raw["has_ratio"]))
print()
print(pd.crosstab(raw["Footnote"], raw["has_disch"]))

has_disch  False  True 
has_ratio              
False       6405    205
True        3683   8037

has_ratio  False  True 
Footnote               
               0  11343
1           3150      0
29             0    377
5           3255      0
7            205      0

has_ratio               False  True 
Number of Readmissions              
False                    6610   8037
True                        0   3683

has_disch  False  True 
Footnote               
            3545   7798
1           3150      0
29           138    239
5           3255      0
7              0    205


In [81]:
numeric_cols = [
    "Predicted Readmission Rate",
    "Expected Readmission Rate",
    "Excess Readmission Ratio",
]

raw[numeric_cols] = raw[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

raw["Calculated Ratio"] = (
    raw["Predicted Readmission Rate"] / raw["Expected Readmission Rate"]
)
raw["Ratio Difference"] = (
    raw["Excess Readmission Ratio"] - raw["Calculated Ratio"]
)

print(raw["Excess Readmission Ratio"].describe())

count    11720.000000
mean         1.001777
std          0.082050
min          0.469800
25%          0.958200
50%          0.997300
75%          1.042300
max          1.629700
Name: Excess Readmission Ratio, dtype: float64
